# Lab 14 — Compare ResNet-18, ResNet-34 and ResNet-50
PneumoniaMNIST transfer-learning comparison. Streamlined Colab edition.

In [ ]:
!pip -q install medmnist
import time,torch,torch.nn as nn,torch.optim as optim,pandas as pd
from torch.utils.data import DataLoader,Subset
from torchvision import transforms,models
import medmnist
from medmnist import INFO
from sklearn.metrics import accuracy_score,f1_score
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
D=getattr(medmnist,INFO['pneumoniamnist']['python_class']); tf=transforms.Compose([transforms.Resize((96,96)),transforms.Grayscale(3),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])]); train=D(split='train',transform=tf,download=True); test=D(split='test',transform=tf,download=True); train=Subset(train,range(min(2000,len(train)))); tr=DataLoader(train,batch_size=32,shuffle=True); te=DataLoader(test,batch_size=64)

In [ ]:
builders={'ResNet-18':(models.resnet18,models.ResNet18_Weights.DEFAULT),'ResNet-34':(models.resnet34,models.ResNet34_Weights.DEFAULT),'ResNet-50':(models.resnet50,models.ResNet50_Weights.DEFAULT)}
rows=[]
for name,(fn,w) in builders.items():
 model=fn(weights=w); [setattr(p,'requires_grad',False) for p in model.parameters()]; model.fc=nn.Linear(model.fc.in_features,2); model=model.to(device); opt=optim.SGD(model.fc.parameters(),lr=.01,momentum=.9); loss_fn=nn.CrossEntropyLoss(); t=time.time()
 for e in range(2):
  model.train()
  for x,y in tr:
   x=x.to(device); y=y.squeeze().long().to(device); opt.zero_grad(); z=model(x); loss=loss_fn(z,y); loss.backward(); opt.step()
 model.eval(); yt=[]; yp=[]
 with torch.no_grad():
  for x,y in te: yt.extend(y.squeeze().numpy()); yp.extend(model(x.to(device)).argmax(1).cpu().numpy())
 rows.append([name,accuracy_score(yt,yp),f1_score(yt,yp),sum(p.numel() for p in model.parameters()),time.time()-t]); del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None
print(pd.DataFrame(rows,columns=['Model','Accuracy','F1','Parameters','Seconds']))